<a href="https://colab.research.google.com/github/poojithavakada7-hue/Railway_Point_Machine/blob/main/Railway_Point_Machine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

from xgboost import XGBClassifier


# ============================================================
# LOAD ORIGINAL 200K DATASET
# ============================================================

file_path = "railway_point_machine_200k.csv"

df = pd.read_csv(file_path)

print("Original Dataset Shape:", df.shape)


# ============================================================
# FEATURES
# ============================================================

features = [
    "Throw_Time_ms",
    "Steady_State_Current_A",
    "Track_Vibration_Hz",
    "Motor_Temp_C",
    "Ambient_Temp_C",
    "Season"
]

target = "Fault_Class"


# ============================================================
# SELECT REQUIRED DATA
# ============================================================

df = df[features + [target]].copy()

# Remove missing values
df = df.dropna()

print("Dataset after cleaning:", df.shape)


# ============================================================
# X AND Y
# ============================================================

X = df[features]
y = df[target]


# ============================================================
# ENCODE TARGET
# ============================================================

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)


print("\n====================================")
print("FAULT CLASSES")
print("====================================")

for i, name in enumerate(label_encoder.classes_):
    print(i, "=", name)


# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("\nTraining samples:", len(X_train))
print("Testing samples :", len(X_test))


# ============================================================
# NUMERICAL FEATURES
# ============================================================

numeric_features = [
    "Throw_Time_ms",
    "Steady_State_Current_A",
    "Track_Vibration_Hz",
    "Motor_Temp_C",
    "Ambient_Temp_C"
]


# ============================================================
# CATEGORICAL FEATURES
# ============================================================

categorical_features = [
    "Season"
]


# ============================================================
# PREPROCESSOR
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            "passthrough",
            numeric_features
        ),

        (
            "season",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)


# ============================================================
# XGBOOST MODEL
# ============================================================

num_classes = len(label_encoder.classes_)

xgb_model = XGBClassifier(

    n_estimators=100,

    max_depth=8,

    learning_rate=0.1,

    subsample=0.9,

    colsample_bytree=0.9,

    objective="multi:softprob",

    num_class=num_classes,

    eval_metric="mlogloss",

    random_state=42,

    n_jobs=-1
)


# ============================================================
# COMPLETE PIPELINE
# ============================================================

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("xgboost", xgb_model)
])


# ============================================================
# TRAIN MODEL
# ============================================================

print("\n====================================")
print("TRAINING XGBOOST")
print("====================================")

pipeline.fit(
    X_train,
    y_train
)

print("Training completed!")


# ============================================================
# PREDICTION
# ============================================================

y_pred = pipeline.predict(X_test)


# ============================================================
# EVALUATION
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("\n====================================")
print("MODEL PERFORMANCE")
print("====================================")

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")


# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\n====================================")
print("CLASSIFICATION REPORT")
print("====================================")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)


# ============================================================
# SAVE COMPLETE MODEL
# ============================================================

model_package = {

    "model": pipeline,

    "label_encoder": label_encoder,

    "feature_columns": features,

    "numeric_features": numeric_features,

    "categorical_features": categorical_features,

    "accuracy": accuracy,

    "precision": precision,

    "recall": recall,

    "f1_score": f1,

    "n_estimators": 100
}


# ============================================================
# SAVE JOBLIB
# ============================================================

model_path = "railway_point_machine_xgboost.joblib"

joblib.dump(
    model_package,
    model_path
)


print("\n====================================")
print("MODEL SAVED SUCCESSFULLY")
print("====================================")

print("File:", model_path)

Original Dataset Shape: (200000, 18)
Dataset after cleaning: (200000, 7)

FAULT CLASSES
0 = F-02: Thickened Lube / Freeze
1 = F-03: Thermal Rail Expansion Lock
2 = F-04: Hard Ballast Obstruction
3 = F-05: Loose Fasteners
4 = F-06: Gearbox Wear (Pitting)
5 = F-07: Broken Connecting Rod
6 = F-08: Facing Point Lock Jam
7 = F-09: Slide Chair Dryness / Dust
8 = F-10: Commutator Brush Wear
9 = F-11: Insulation Short (Water Ingress)
10 = F-15: Ballast Washout (Misalignment)
11 = F-16: Ice Blockage in Switch
12 = Healthy
13 = Idle

Training samples: 160000
Testing samples : 40000

TRAINING XGBOOST
Training completed!

MODEL PERFORMANCE
Accuracy  : 0.9531
Precision : 0.9507
Recall    : 0.9531
F1 Score  : 0.9499

CLASSIFICATION REPORT
                                        precision    recall  f1-score   support

         F-02: Thickened Lube / Freeze       0.95      0.99      0.97       493
     F-03: Thermal Rail Expansion Lock       0.78      0.89      0.84       484
        F-04: Hard Balla

In [ ]:
import gradio as gr
import pandas as pd
import numpy as np
import joblib


# ============================================================
# LOAD MODEL
# ============================================================

model_package = joblib.load(
    "railway_point_machine_xgboost.joblib"
)

model = model_package["model"]

label_encoder = model_package["label_encoder"]


# ============================================================
# PREDICTION FUNCTION
# ============================================================

def predict_fault(
    throw_time,
    current,
    vibration,
    motor_temp,
    ambient_temp,
    season
):

    # --------------------------------------------------------
    # CREATE INPUT DATAFRAME
    # --------------------------------------------------------

    input_data = pd.DataFrame({

        "Throw_Time_ms": [throw_time],

        "Steady_State_Current_A": [current],

        "Track_Vibration_Hz": [vibration],

        "Motor_Temp_C": [motor_temp],

        "Ambient_Temp_C": [ambient_temp],

        "Season": [season]

    })


    # --------------------------------------------------------
    # PREDICTION
    # --------------------------------------------------------

    prediction = model.predict(
        input_data
    )


    # --------------------------------------------------------
    # CONVERT CLASS NUMBER TO FAULT NAME
    # --------------------------------------------------------

    fault_name = label_encoder.inverse_transform(
        prediction.astype(int)
    )[0]


    # --------------------------------------------------------
    # PREDICTION PROBABILITIES
    # --------------------------------------------------------

    probabilities = model.predict_proba(
        input_data
    )[0]


    classes = label_encoder.classes_


    probability_dict = {

        classes[i]: float(probabilities[i])

        for i in range(len(classes))

    }


    # --------------------------------------------------------
    # CONFIDENCE
    # --------------------------------------------------------

    confidence = float(
        max(probabilities) * 100
    )


    # --------------------------------------------------------
    # RETURN
    # --------------------------------------------------------

    return (

        fault_name,

        f"{confidence:.2f}%",

        probability_dict

    )


# ============================================================
# GRADIO INTERFACE
# ============================================================

interface = gr.Interface(

    fn=predict_fault,


    # ========================================================
    # INPUTS
    # ========================================================

    inputs=[

        gr.Number(
            label="Throw Time (ms)",
            value=100
        ),


        gr.Number(
            label="Steady-State Current (A)",
            value=5
        ),


        gr.Number(
            label="Track Vibration (Hz)",
            value=10
        ),


        gr.Number(
            label="Motor Temperature (°C)",
            value=40
        ),


        gr.Number(
            label="Ambient Temperature (°C)",
            value=25
        ),


        # IMPORTANT:
        # These values exactly match the dataset

        gr.Dropdown(

            choices=[

                "Peak Summer",

                "Monsoon",

                "Spring/Autumn",

                "Harsh Winter"

            ],

            value="Monsoon",

            label="Season"

        )

    ],


    # ========================================================
    # OUTPUTS
    # ========================================================

    outputs=[

        gr.Textbox(
            label="Predicted Fault"
        ),


        gr.Textbox(
            label="Confidence"
        ),


        gr.Label(
            label="Fault Probabilities"
        )

    ],


    # ========================================================
    # TITLE
    # ========================================================

    title="🚆 Railway Point Machine Fault Detection",


    description=(
        "XGBoost-based railway point machine "
        "fault classification system"
    ),


    flagging_mode="never"

)


# ============================================================
# LAUNCH
# ============================================================

interface.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f4eacb51f8b70750cd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
